In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.src.applications.mobilenet_v3 import preprocess_input
import keras
from keras import layers
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Data paths and parameters
train_dir = '/home/groggy/data/a-large-scale-fish-dataset/2/Fish_Dataset/Fish_Dataset_clean'
img_size = (224, 224)
batch_size = 32
project_root = Path('/home/groggy/projects/Image-Classification-Explainability')

In [ ]:
train_datagen = ImageDataGenerator(
    rescale = 1.0 / 255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

num_classes = train_generator.num_classes
print(f"Number of classes: {num_classes}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

In [ ]:
inputs = keras.Input(shape=(224, 224, 3))

x = layers.Conv2D(64, 3, activation="relu")(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)

x = layers.Conv2D(128, 3, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)

x = layers.Conv2D(256, 3, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)

x = layers.Conv2D(512, 3, activation="relu")(x)
x = layers.BatchNormalization()(x)

x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)

outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

model = Model(inputs=inputs, outputs=outputs)

print(f"\\nTotal parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print(f"Non-trainable parameters: {sum([tf.size(w).numpy() for w in model.non_trainable_weights]):,}")

In [ ]:
# Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

In [ ]:
# Visualize training results
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.axhline(y=0.2589, color='r', linestyle='--', label='Baseline (25.89%)')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Results summary
best_val_acc = max(history.history['val_accuracy'])
final_val_acc = history.history['val_accuracy'][-1]
baseline_acc = 0.2589

print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"Baseline validation accuracy:     {baseline_acc:.4f} (25.89%)")
print(f"Current best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
print(f"Final validation accuracy:        {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
print(f"Improvement over baseline:        {(best_val_acc - baseline_acc):.4f} ({(best_val_acc - baseline_acc)*100:.2f}%)")
print("=" * 60)

if best_val_acc > baseline_acc:
    print("✓ Architecture improved performance!")
else:
    print("✗ No improvement - try different architecture")

In [ ]:
# Save model if improved
if best_val_acc > baseline_acc:
    save_path = project_root / 'ML' / 'Models' / 'MobileNetV3Large_Improved.keras'
    model.save(save_path)
    print(f'✓ Model saved to: {save_path}')
else:
    print('Model not saved (no improvement over baseline)')

In [ ]:
model.summary()